# CIFAR-10: direct sequential ARL0 calibration 

This notebook reuses the validated 70%-CPV static-PCA pipeline but replaces score-block calibration with an observation-level sequential bootstrap. Each in-control bootstrap stream is sampled before window construction; the complete overlapping-window detector path is then recomputed from its zero state. All detector configurations share the same streams. The calibration horizon is ten times the target score-step ARL, and calibration and held-out tables include Monte Carlo confidence intervals and censoring diagnostics.

EWMA is evaluated at the prespecified lambdas 0.05, 0.10, 0.20, 0.40, and 1.00. Phase II uses exactly one paired episode per seed and condition; ARL0 calibration retains its much larger simulation count. Severity 1.00 is enabled by default. Change `SEVERITY_LEVELS` to include other levels. Compatible VAE checkpoints are copied forward from earlier output roots, so the VAE is not retrained when a matching checkpoint exists.


## Load the validated pipeline and direct calibration extension


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import os

DATASET = "cifar10"
QUICK_RUN = os.environ.get("DIRECT_ARL0_QUICK", "0") == "1"
RUN_DIRECT = os.environ.get("DIRECT_ARL0_RUN_FULL", "1") == "1"
SEVERITY_LEVELS = (1.00,)
EWMA_LAMBDAS = (0.05, 0.10, 0.20, 0.40, 1.00)

folder_candidate = Path.home() / "Documents" / "Stats Thesis notebooks and results" / "02_observation_level_calibration_extension"
PACKAGE_ROOT = next((root for root in (Path.cwd(), folder_candidate)
                     if (root / "observation_level_sequential_calibration.py").is_file()), Path.cwd())
BASE_ROOT = PACKAGE_ROOT
SOURCE_NOTEBOOK = BASE_ROOT / "cifar10_cpv70_static_pca_base.ipynb"
EXTENSION = PACKAGE_ROOT / "observation_level_sequential_calibration.py"
if not SOURCE_NOTEBOOK.is_file() or not EXTENSION.is_file():
    raise FileNotFoundError("Keep this public-code folder beside the CPV-based source notebooks.")

os.environ["DRIFT_RUN_FULL"] = "0"
os.environ["DRIFT_QUICK"] = "1" if QUICK_RUN else "0"
os.environ["DRIFT_RUN_BASE"] = str(BASE_ROOT)
document = json.loads(SOURCE_NOTEBOOK.read_text())
for cell_index, cell in enumerate(document["cells"]):
    if cell_index > 24:
        break
    if cell.get("cell_type") != "code":
        continue
    source = "".join(cell.get("source", []))
    source = "\n".join(line for line in source.splitlines()
                       if not line.lstrip().startswith(("%", "!")))
    exec(compile(source, f"{SOURCE_NOTEBOOK.name}:cell{cell_index}", "exec"), globals())

OUTPUT_ROOT = (
    BASE_ROOT / "outputs/observation_level_calibration_static"
    / DATASET / "static_pca"
)
calibration_episodes = 20 if QUICK_RUN else int(os.environ.get("DIRECT_ARL0_CAL_EPISODES", "100"))
heldout_episodes = 20 if QUICK_RUN else int(os.environ.get("DIRECT_ARL0_TEST_EPISODES", "25"))
CFG = replace(
    CFG,
    out_root=str(OUTPUT_ROOT),
    severity_levels=SEVERITY_LEVELS,
    ewma_lambda=0.20,
    ewma_lambda_grid=EWMA_LAMBDAS,
    mc_arl1_reps=1,
    arl0_cal_episodes=calibration_episodes,
    arl0_test_episodes=heldout_episodes,
    arl0_horizon_mult=10,
    force_recompute=os.environ.get("DIRECT_ARL0_FORCE_RECOMPUTE", "0") == "1",
)
CFG.legacy_out_roots = ()
CFG.vae_legacy_out_roots = (
    str(BASE_ROOT / "outputs/cifar10_static_pca_benchmark"),
    str(BASE_ROOT / "outputs/cifar10_cpv70_static_pca"),
    str(BASE_ROOT / "outputs/cpv70_ewma_primary" / DATASET / "static_pca"),
    str(BASE_ROOT / "outputs/observation_level_ewma_primary" / DATASET / "static_pca"),
)
Path(CFG.out_root, "aggregate").mkdir(parents=True, exist_ok=True)
_PROGRESS_PATH = Path(CFG.out_root) / "progress.json"
exec(compile(EXTENSION.read_text(), str(EXTENSION), "exec"), globals())

assert CFG.ewma_lambda_grid == (0.05, 0.10, 0.20, 0.40, 1.00)
assert CFG.mc_arl1_reps == 1
assert CFG.severity_levels == (1.00,)
assert CFG.arl0_horizon_mult == 10
print(f"dataset={DATASET}")
print(f"output root={CFG.out_root}")
print(f"final Phase-II episodes per seed/condition={CFG.mc_arl1_reps}")
print(f"calibration/held-out episodes={CFG.arl0_cal_episodes}/{CFG.arl0_test_episodes}")
print(f"calibration horizon={observation_horizon(CFG)} observations")


## Execute


In [ ]:
if RUN_DIRECT:
    _results, _heldout = [], []
    for _seed in CFG.seeds:
        _results.append(evaluate_seed(_seed, CFG, EXTRACTOR))
        _heldout.append(verify_heldout_arl0(_seed, CFG, EXTRACTOR))
    RESULTS = pd.concat(_results, ignore_index=True)
    HELDOUT_ARL0 = pd.concat(_heldout, ignore_index=True)
    AGGREGATES = aggregate_results(RESULTS, HELDOUT_ARL0, CFG)
else:
    print("Configuration validated. Set DIRECT_ARL0_RUN_FULL=1 to execute.")


## Design audit


In [ ]:
display(pd.DataFrame([{
    "dataset": CFG.dataset_name,
    "calibration_method": DIRECT_CALIBRATION_METHOD,
    "bootstrap_unit": "in_control_observation",
    "arl0_start": "zero_state",
    "target_arl0_observations": CFG.target_arl0_observations,
    "calibration_episodes": CFG.arl0_cal_episodes,
    "heldout_episodes": CFG.arl0_test_episodes,
    "calibration_horizon_observations": observation_horizon(CFG),
    "pca_target_cpv": CFG.pca_retention,
    "ewma_lambdas": CFG.ewma_lambda_grid,
    "phase2_episodes_per_seed_condition": CFG.mc_arl1_reps,
    "severity_levels": CFG.severity_levels,
    "output_root": CFG.out_root,
    "vae_policy": "copy compatible prior checkpoint; do not retrain when available",
}]))
